# BANK MARKETING CAMPAIGN — EDA, Classification & Customer Segmentation

**Phiên bản chỉnh sửa** từ notebook đã gửi. Mục tiêu: (1) khám phá dữ liệu, (2) dự đoán khách hàng đăng ký tiền gửi (`y`), (3) phân khúc theo **nhân khẩu học** bằng K-means.

**Các thay đổi chính:** EDA đầy đủ; train/validation/test độc lập; cân bằng dữ liệu **chỉ trong train / từng fold CV**; chọn threshold trên validation; so sánh Logistic Regression / Random Forest / XGBoost; K-means chỉ dựa vào `age`, `job`, `marital`, `education`, mô tả từng cụm.

> Không xóa/capping outlier theo lựa chọn của người thực hiện. `duration` bị loại khỏi phân loại để tránh dùng thông tin chưa biết trước cuộc gọi. **Không dùng nhãn `y` để phân cụm.** Hãy chạy **Runtime → Run all** theo thứ tự từ trên xuống.

## 0. Thư viện và đọc dữ liệu

In [ ]:
# Nếu thiếu thư viện trên Colab, bỏ dấu # ở dòng bên dưới rồi chạy một lần:
# %pip install -q imbalanced-learn xgboost

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    silhouette_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
# Ưu tiên đường dẫn Google Drive nếu tồn tại; nếu không tìm file ở thư mục làm việc.
# Tùy chỉnh DRIVE_PATH nếu thư mục trên Drive của anh khác.
DRIVE_PATH = "/content/drive/MyDrive/Data_Science/bank-additional-full.csv"
LOCAL_CANDIDATES = [
    "bank-additional-full.csv",
    "/content/bank-additional-full.csv",
    "/mnt/data/bank-additional-full.csv",
]

csv_path = next((p for p in [DRIVE_PATH] + LOCAL_CANDIDATES if os.path.isfile(p)), None)
if csv_path is None:
    try:
        from google.colab import files
        print("Không tìm thấy CSV. Hãy upload bank-additional-full.csv:")
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("Hãy đặt bank-additional-full.csv cùng thư mục notebook hoặc sửa DRIVE_PATH.")

df = pd.read_csv(csv_path, sep=";")
print("Đường dẫn:", csv_path)
print("Kích thước:", df.shape)
display(df.head())

## 1. EDA — Phân tích khám phá dữ liệu

Phân tích trên dữ liệu **gốc**, trước khi loại `duration` để không làm mất thông tin phục vụ EDA. Dữ liệu gốc có thể không có `NaN` nhưng chứa giá trị chuỗi `unknown` và mã đặc biệt `pdays=999`.

In [ ]:
print("Cấu trúc dữ liệu:")
df.info()
print("\nThống kê dữ liệu số:")
display(df.describe().T)
print("\nThống kê dữ liệu phân loại:")
display(df.describe(include="object").T)
print("\nSố dòng trùng hoàn toàn:", df.duplicated().sum())

quality = pd.DataFrame({
    "missing_na": df.isna().sum(),
    "unknown": (df == "unknown").sum(),
    "unique_values": df.nunique()
})
quality["unknown_pct"] = (quality["unknown"] / len(df) * 100).round(2)
display(quality)
print("pdays=999 (không có liên hệ trước):", (df["pdays"] == 999).sum())

In [ ]:
# Phân bố nhãn: chú ý mất cân bằng lớp
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="y", order=["no", "yes"], color="steelblue", ax=ax[0])
ax[0].set_title("Số lượng khách hàng theo y")
counts = df["y"].value_counts().reindex(["no", "yes"])
ax[1].pie(counts, labels=counts.index, autopct="%1.2f%%", startangle=90)
ax[1].set_title("Tỷ lệ đăng ký tiền gửi")
plt.tight_layout(); plt.show()
print("Tỷ lệ nhãn (%):")
display((df["y"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))

In [ ]:
# Histogram và Boxplot các biến số (không tự động xóa ngoại lai)
numeric_eda = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(5, 2, figsize=(14, 17))
for ax, col in zip(axes.flat, numeric_eda):
    sns.histplot(data=df, x=col, bins=30, color="steelblue", ax=ax)
    ax.set_title(f"Histogram: {col}")
for ax in list(axes.flat)[len(numeric_eda):]: ax.axis("off")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(5, 2, figsize=(14, 13))
for ax, col in zip(axes.flat, numeric_eda):
    sns.boxplot(data=df, x=col, color="steelblue", ax=ax)
    ax.set_title(f"Boxplot: {col}")
for ax in list(axes.flat)[len(numeric_eda):]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Countplot cho biến phân loại (loại y)
categorical_eda = [c for c in df.select_dtypes(include=["object", "category"]).columns if c != "y"]
fig, axes = plt.subplots(5, 2, figsize=(16, 21))
for ax, col in zip(axes.flat, categorical_eda):
    sns.countplot(data=df, y=col, order=df[col].value_counts().index, color="steelblue", ax=ax)
    ax.set_title(f"Countplot: {col}")
for ax in list(axes.flat)[len(categorical_eda):]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Quan hệ với target: tỷ lệ đăng ký yes trong từng nhóm
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, ["job", "marital", "education", "poutcome"]):
    rates = (df.groupby(col)["y"].apply(lambda s: s.eq("yes").mean() * 100)
             .sort_values(ascending=False))
    sns.barplot(x=rates.values, y=rates.index, color="steelblue", ax=ax)
    ax.set_xlabel("Tỷ lệ đăng ký (%)"); ax.set_title(f"Tỷ lệ đăng ký theo {col}")
plt.tight_layout(); plt.show()

# Tương quan Pearson giữa các biến số (không coi đây là quan hệ nhân quả)
corr_df = df.select_dtypes(include=np.number).copy()
corr_df["y_binary"] = df["y"].eq("yes").astype(int)
plt.figure(figsize=(12, 9))
sns.heatmap(corr_df.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap (chỉ các biến số)"); plt.tight_layout(); plt.show()

**Nhận xét EDA:** Nhãn `yes` ít hơn nhãn `no`, nên không dựa duy nhất vào Accuracy. `unknown` là một danh mục chưa rõ, chưa tự xóa. `pdays=999` là mã *không từng liên hệ trong chiến dịch trước*, không phải số ngày thực tế. Boxplot chỉ đánh dấu ngoại lai theo thống kê; chúng ta **giữ nguyên** ngoại lai hợp lệ. Các khác biệt tỷ lệ giữa nhóm là **liên hệ quan sát được**, không phải quan hệ nhân quả.

## 2. Dữ liệu đầu vào, tách Train / Validation / Test

In [ ]:
# Mục tiêu dự đoán trước cuộc gọi: không dùng duration (chỉ biết sau cuộc gọi)
X = df.drop(columns=["y", "duration"]).copy()
y = df["y"].map({"no": 0, "yes": 1}).astype(int)

# pdays=999 có ý nghĩa đặc biệt: tạo cờ rồi thay 999 bằng NaN
X["previously_contacted"] = (X["pdays"] != 999).astype(int)
X["pdays"] = X["pdays"].replace(999, np.nan)

# 80% train+validation; 20% test giữ nguyên hoàn toàn để đánh giá cuối
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
# 20% của Train-full làm validation -> 64% train / 16% val / 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    stratify=y_train_full, random_state=RANDOM_STATE
)

for name, yy in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: {len(yy):,} dòng; tỷ lệ yes = {yy.mean():.2%}")

## 3. Data Transformation — Encoding và Scaling (không fit trên Test)

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# OneHotEncoder giữ unknown là một nhóm nếu xuất hiện trên Train
num_pipeline = SkPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = SkPipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Những lệnh này để kiểm tra trực quan transformation; khi CV, pipeline bên dưới
# sẽ fit preprocessor riêng theo từng fold, tránh leakage.
X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
X_train_final = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_val_final = pd.DataFrame(X_val_transformed, columns=feature_names, index=X_val.index)
X_test_final = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)
print("Numerical:", num_cols)
print("Categorical:", cat_cols)
print("Shape sau mã hóa:", X_train_final.shape, X_val_final.shape, X_test_final.shape)

## 4. Xử lý mất cân bằng đúng cách

**Không SMOTE sau One-Hot** vì có thể sinh giá trị thập phân trong các cột phân loại nhị phân. Ở đây sử dụng `RandomOverSampler` bên trong `imblearn.Pipeline`, sao cho chỉ tập Train của mỗi lượt fit / fold được lấy mẫu bổ sung. Tập Validation và Test **giữ nguyên tỷ lệ nhãn**. Ngoài ra so sánh phương án `class_weight="balanced"` không oversampling.

In [ ]:
print("Phân bố nhãn trên Train gốc:")
display(y_train.value_counts().rename_axis("class").to_frame("count"))

# Pipeline tham chiếu: fit encoder, oversampler, model theo đúng thứ tự
rf_ros_pipeline = ImbPipeline([
    ("preprocess", preprocessor),
    ("balance", RandomOverSampler(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(n_estimators=160, min_samples_leaf=3,
                                      random_state=RANDOM_STATE, n_jobs=-1))
])
print("Pipeline:", [name for name, _ in rf_ros_pipeline.steps])

## 5. Train các mô hình trên Train; chọn Threshold trên Validation

In [ ]:
# Hàm chọn threshold: tối đa hóa F1 lớp 1 trên VALIDATION, không đụng Test
threshold_grid = np.arange(0.10, 0.901, 0.02)

def find_best_threshold(model, X_valid, y_valid):
    prob = model.predict_proba(X_valid)[:, 1]
    rows = []
    for t in threshold_grid:
        pred = (prob >= t).astype(int)
        rows.append({
            "threshold": float(round(t, 2)),
            "precision_1": precision_score(y_valid, pred, zero_division=0),
            "recall_1": recall_score(y_valid, pred, zero_division=0),
            "f1_1": f1_score(y_valid, pred, zero_division=0),
            "accuracy": accuracy_score(y_valid, pred)
        })
    table = pd.DataFrame(rows)
    best = table.sort_values(["f1_1", "accuracy"], ascending=False).iloc[0]
    return float(best["threshold"]), table

# Dùng cùng một cách chọn threshold trên validation cho 3 mô hình
models = {
    "Logistic Regression (class_weight)": ImbPipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1500, class_weight="balanced",
                                      random_state=RANDOM_STATE))
    ]),
    "Random Forest (ROS)": rf_ros_pipeline,
    "Random Forest (class_weight)": ImbPipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=160, min_samples_leaf=3,
                                           class_weight="balanced",
                                           random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "XGBoost (scale_pos_weight)": ImbPipeline([
        ("preprocess", preprocessor),
        ("model", XGBClassifier(n_estimators=220, max_depth=4, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8,
                                 scale_pos_weight=(y_train.eq(0).sum() / y_train.eq(1).sum()),
                                 objective="binary:logistic", eval_metric="logloss",
                                 random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

validation_tables = {}
selected_thresholds = {}
for name, model in models.items():
    print("\nĐang train:", name)
    model.fit(X_train, y_train)
    t, table = find_best_threshold(model, X_val, y_val)
    validation_tables[name] = table
    selected_thresholds[name] = t
    chosen = table.loc[table["threshold"].eq(t)].iloc[0]
    print(f"Threshold (val): {t:.2f}; F1 lớp 1 (val): {chosen['f1_1']:.4f}; "
          f"Recall: {chosen['recall_1']:.4f}; Accuracy: {chosen['accuracy']:.4f}")

val_summary = pd.DataFrame([
    {"Model": name, "Threshold": t,
     **validation_tables[name].loc[validation_tables[name]["threshold"].eq(t),
                                   ["precision_1", "recall_1", "f1_1", "accuracy"]].iloc[0].to_dict()}
    for name, t in selected_thresholds.items()
]).sort_values("f1_1", ascending=False)
display(val_summary)

In [ ]:
# Minh họa đường Precision / Recall / F1 theo threshold trên Validation
name_to_plot = "Random Forest (ROS)"
tab = validation_tables[name_to_plot]
plt.figure(figsize=(10, 5))
for metric in ["precision_1", "recall_1", "f1_1"]:
    plt.plot(tab["threshold"], tab[metric], label=metric)
plt.axvline(selected_thresholds[name_to_plot], linestyle="--", color="black",
            label=f"Selected: {selected_thresholds[name_to_plot]:.2f}")
plt.title(f"Threshold tuning trên Validation — {name_to_plot}")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.legend(); plt.tight_layout(); plt.show()

### 5.1. Hyperparameter tuning (tùy chọn, có Cross-Validation đúng quy trình)

Cell sau thực hiện `RandomizedSearchCV` trên **Train** với oversampling và preprocessing nằm trong pipeline. Không dùng Validation/Test khi tìm tham số. `n_iter=8, cv=3` để notebook dễ chạy trên Colab; tăng lên nếu muốn khảo sát kỹ hơn.

In [ ]:
# Có thể chỉnh RUN_TUNING=False để chạy nhanh; mặc định True để có minh chứng CV.
RUN_TUNING = True
if RUN_TUNING:
    rf_cv_pipe = ImbPipeline([
        ("preprocess", preprocessor),
        ("balance", RandomOverSampler(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
    ])
    params = {
        "model__n_estimators": [100, 160, 220],
        "model__max_depth": [10, 18, None],
        "model__min_samples_leaf": [1, 3, 5],
        "model__min_samples_split": [2, 5, 10],
    }
    rf_search = RandomizedSearchCV(
        rf_cv_pipe, param_distributions=params, n_iter=8, cv=3,
        scoring="f1", random_state=RANDOM_STATE, n_jobs=-1, verbose=1
    )
    rf_search.fit(X_train, y_train)
    tuned_rf = rf_search.best_estimator_
    tuned_t, tuned_tab = find_best_threshold(tuned_rf, X_val, y_val)
    models["Random Forest (ROS + CV tuned)"] = tuned_rf
    selected_thresholds["Random Forest (ROS + CV tuned)"] = tuned_t
    validation_tables["Random Forest (ROS + CV tuned)"] = tuned_tab
    print("Tuned RF params:", rf_search.best_params_)
    print("Best CV F1 at default threshold:", round(rf_search.best_score_, 4))
    print("Chosen Validation threshold:", tuned_t)
else:
    print("Bỏ qua RandomizedSearchCV để chạy nhanh.")

## 6. Đánh giá cuối cùng trên Test (không tối ưu lại bằng Test)

Chốt mô hình và threshold qua Validation, sau đó **fit lại trên Train+Validation** và đánh giá một lần trên Test. Kết quả dưới đây được tính khi chạy notebook, không tự khẳng định điểm số trước.

In [ ]:
from sklearn.base import clone

# Không thay đổi selected_thresholds sau khi xem Test.
final_models = {}
comparison_rows = []
all_predictions = {}

for name, model in models.items():
    final_model = clone(model)
    # XGBoost weight được tính lại theo Train-full khi refit, không sử dụng Test
    if name == "XGBoost (scale_pos_weight)":
        final_model.set_params(model__scale_pos_weight=(y_train_full.eq(0).sum() / y_train_full.eq(1).sum()))
    final_model.fit(X_train_full, y_train_full)
    final_models[name] = final_model

    threshold = selected_thresholds[name]
    prob = final_model.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)
    all_predictions[name] = (pred, prob)
    comparison_rows.append({
        "Model": name, "Threshold (from Val)": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_1": precision_score(y_test, pred, zero_division=0),
        "Recall_1": recall_score(y_test, pred, zero_division=0),
        "F1_1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("F1_1", ascending=False)
display(comparison_df.round(4))

for name, (pred, prob) in all_predictions.items():
    print("\n=========", name, "=========")
    print(classification_report(y_test, pred, digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=[0, 1], display_labels=["No (0)", "Yes (1)"],
        cmap="Blues", values_format="d"
    )
    plt.title(f"{name} | threshold = {selected_thresholds[name]:.2f}")
    plt.tight_layout(); plt.show()

## 7. Feature Importance — XGBoost

Chỉ đo mức đóng góp của đặc trưng **trong mô hình**, **không chứng minh quan hệ nhân quả** (không thể khẳng định lãi suất tăng sẽ làm khách hàng đăng ký do biểu đồ importance).

In [ ]:
xgb_final_pipeline = final_models["XGBoost (scale_pos_weight)"]
xgb_estimator = xgb_final_pipeline.named_steps["model"]
feature_names_xgb = xgb_final_pipeline.named_steps["preprocess"].get_feature_names_out()
importance_df = pd.DataFrame({
    "Feature": feature_names_xgb,
    "Importance": xgb_estimator.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))
plt.figure(figsize=(11, 6))
sns.barplot(data=importance_df.head(15), x="Importance", y="Feature", color="steelblue")
plt.title("Top 15 đặc trưng theo XGBoost feature importance")
plt.tight_layout(); plt.show()

### Nhận xét Feature Importance (cách viết an toàn)

Sau khi chạy cell trên, anh điền **tên các biến thực sự nằm trong Top 10** vào báo cáo. Những biến như `euribor3m`, `poutcome`, `previously_contacted` (nếu xuất hiện) có thể hữu ích cho khả năng **dự đoán** của mô hình trong dữ liệu lịch sử. Không khẳng định chúng **gây ra** hành vi gửi tiền; có thể tồn tại yếu tố khác hoặc sự phụ thuộc vào thời điểm chiến dịch. So sánh mức đóng góp không đồng nghĩa chiều tác động (tăng/giảm xác suất).

## 8. Phân khúc khách hàng theo nhân khẩu học bằng K-means

**Chỉ sử dụng `age`, `job`, `marital`, `education`**. Không sử dụng dữ liệu SMOTE, không sử dụng `y`, chiến dịch hay yếu tố kinh tế để xây dựng cụm. `y` chỉ dùng **sau phân cụm** để phân tích tỷ lệ đăng ký. Dữ liệu `unknown` được giữ như một danh mục riêng.

In [ ]:
cluster_features = ["age", "job", "marital", "education"]
X_cluster = df[cluster_features].copy()

cluster_preprocessor = ColumnTransformer([
    ("age", StandardScaler(), ["age"]),
    ("demographic", OneHotEncoder(handle_unknown="ignore"),
     ["job", "marital", "education"])
])
X_cluster_scaled = cluster_preprocessor.fit_transform(X_cluster)
print("Đặc trưng phân cụm:", cluster_features)
print("Shape sau biến đổi:", X_cluster_scaled.shape)

In [ ]:
# Elbow + Silhouette (Silhouette ước lượng trên mẫu để tiết kiệm thời gian)
k_values = list(range(2, 9))
cluster_metrics = []
for k in k_values:
    model_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = model_k.fit_predict(X_cluster_scaled)
    score_k = silhouette_score(X_cluster_scaled, labels_k,
                               sample_size=1500, random_state=RANDOM_STATE)
    cluster_metrics.append({"K": k, "Inertia": model_k.inertia_,
                            "Silhouette (sample)": score_k})
metrics_df = pd.DataFrame(cluster_metrics)
display(metrics_df.round(4))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(metrics_df["K"], metrics_df["Inertia"], marker="o")
axes[0].set_title("Elbow Method"); axes[0].set_xlabel("K"); axes[0].set_ylabel("Inertia")
axes[1].plot(metrics_df["K"], metrics_df["Silhouette (sample)"], marker="o")
axes[1].set_title("Sample Silhouette Score"); axes[1].set_xlabel("K")
plt.tight_layout(); plt.show()

**Lựa chọn K:** Để bám sát bài gốc, chọn **K = 5** để mô tả năm phân khúc; không khẳng định đây là K có Silhouette cao nhất. Có thể đổi `k_optimal` sau khi xem Elbow/Silhouette và yêu cầu giảng viên.

In [ ]:
k_optimal = 5
kmeans_model = KMeans(n_clusters=k_optimal, n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans_model.fit_predict(X_cluster_scaled)
df_cluster = df[cluster_features + ["y"]].copy()
df_cluster["Cluster"] = cluster_labels

# Bảng tổng quan theo từng cụm
cluster_profile = df_cluster.groupby("Cluster").agg(
    So_khach_hang=("age", "size"),
    Tuoi_TB=("age", "mean"),
    Tuoi_trung_vi=("age", "median"),
    Nghe_pho_bien=("job", lambda s: s.value_counts().index[0]),
    Hon_nhan_pho_bien=("marital", lambda s: s.value_counts().index[0]),
    Hoc_van_pho_bien=("education", lambda s: s.value_counts().index[0]),
    Ty_le_dang_ky=("y", lambda s: s.eq("yes").mean() * 100)
)
cluster_profile["Ty_trong_%"] = cluster_profile["So_khach_hang"] / len(df_cluster) * 100
cluster_profile = cluster_profile[["So_khach_hang", "Ty_trong_%", "Tuoi_TB", "Tuoi_trung_vi",
                                   "Nghe_pho_bien", "Hon_nhan_pho_bien", "Hoc_van_pho_bien",
                                   "Ty_le_dang_ky"]]
display(cluster_profile.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df_cluster, x="Cluster", color="steelblue", ax=axes[0])
axes[0].set_title("Quy mô các cụm")
sns.barplot(x=cluster_profile.index, y=cluster_profile["Ty_le_dang_ky"], color="steelblue", ax=axes[1])
axes[1].set_title("Tỷ lệ đăng ký (%) theo cụm")
axes[1].set_ylabel("Yes (%)")
plt.tight_layout(); plt.show()

In [ ]:
# Phân bố các danh mục trong từng cụm, theo % nội bộ cụm
for col in ["job", "marital", "education"]:
    print(f"\n===== CƠ CẤU {col.upper()} THEO CỤM (%) =====")
    tab = pd.crosstab(df_cluster["Cluster"], df_cluster[col], normalize="index") * 100
    display(tab.round(1))

# Mô tả tự động các cụm dựa trên dữ liệu thực tế (không tự đặt tên thiếu căn cứ)
print("\n===== MÔ TẢ 5 PHÂN KHÚC =====")
for cid, sub in df_cluster.groupby("Cluster"):
    desc = [f"Cụm {cid}: {len(sub):,} khách ({len(sub)/len(df_cluster):.1%})",
            f"tuổi trung bình {sub['age'].mean():.1f}",
            f"tỷ lệ đăng ký {sub['y'].eq('yes').mean():.1%}"]
    print("; ".join(desc))
    for col, label in [("job", "Nghề nghiệp"), ("marital", "Hôn nhân"), ("education", "Học vấn")]:
        top = sub[col].value_counts(normalize=True).head(2)
        print(f"  {label}: " + ", ".join(f"{x} ({p:.1%})" for x, p in top.items()))

### Diễn giải kết quả K-means trong báo cáo

Dựa trên **bảng `cluster_profile` và tỷ trọng các nhóm** được in ở trên, mô tả từng cụm theo: quy mô, tuổi trung bình, nghề nghiệp, hôn nhân, học vấn nổi bật và tỷ lệ `y=yes`. Không đặt tên nhóm trước khi xem phân bố thực tế. Một cụm có tỷ lệ đăng ký cao **không chứng minh** nhân khẩu học gây ra kết quả đăng ký. Do khoảng cách Euclidean sau One-Hot là phương pháp đơn giản, các cụm có thể chồng lấn; xem Silhouette để đánh giá hạn chế. **K-means không sử dụng SMOTE và không đánh giá theo Accuracy/F1.**

## 9. Kết luận và lưu kết quả

- **Classification:** chọn mô hình bằng F1 trên **Validation**, cố định Threshold, sau đó đánh giá trên Test. F1 của lớp 1 nên được xem cùng Precision, Recall, Accuracy và ROC-AUC.
- **Imbalance:** đã so sánh oversampling trong pipeline với `class_weight` / `scale_pos_weight`, không cần kết hợp chúng tùy tiện.
- **Clustering:** năm cụm K-means được xây dựng hoàn toàn bằng nhân khẩu học; `y` chỉ dùng để mô tả hậu nghiệm.
- **Ngoại lai:** giữ nguyên theo yêu cầu; `pdays=999` được xử lý riêng trong Classification.
- **Giới hạn:** Đây là dữ liệu lịch sử/quan sát; không suy diễn quan hệ nhân quả từ Feature Importance hay các tỷ lệ theo cụm.

In [ ]:
# Xuất bảng kết quả để đưa vào báo cáo (tạo trong môi trường đang chạy)
comparison_df.to_csv("bank_model_comparison.csv", index=False)
cluster_profile.to_csv("bank_kmeans_5_cluster_profile.csv", encoding="utf-8-sig")
print("Đã lưu: bank_model_comparison.csv và bank_kmeans_5_cluster_profile.csv")